In [1]:
import scipy.io as sio
import pandas as pd
import numpy as np
import sys, os
import torch
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import Extract_Features

In [2]:
phase1_data = sio.loadmat('../data/mine_impact_data_2019.mat')
samples  = pd.DataFrame(phase1_data["x"].T)
labels  = pd.DataFrame(phase1_data["y"].T, columns=["y"])

df = pd.concat([samples, labels], axis=1, join="inner")

df = df.dropna()

In [3]:

shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data = Extract_Features(
    df_X=df_X,
    df_Y=df_Y,
    feature = "rms_energy",
    frame_size = 128,
    hop_length = 32
)


print(data.get_samples().shape)
print(data.get_labels().shape)



(3309, 1126)
(3309,)


In [4]:

data_zcr = Extract_Features(
    df_X=df_X,
    df_Y=df_Y,
    feature = "zero_crossing_rate",
    frame_size = 128,
    hop_length = 32
)

print(data_zcr.get_samples().shape)
print(data_zcr.get_labels().shape)

(3309, 1126)
(3309,)


In [5]:
from sklearn.svm import SVC, LinearSVC

svc = SVC(kernel="rbf", C=100)
svc.fit(data.get_samples()[:3000], data.get_labels()[:3000])
print(svc.score(data.get_samples()[3000:], data.get_labels()[3000:]))

0.7799352750809061


In [7]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="envelope.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-2, optim="adam", epochs=20)

loops.test(model_path="envelope.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.578642, Train accuracy: 0.6630
[INFO] EPOCH: 2/20
Train loss: 0.516235, Train accuracy: 0.7260
[INFO] EPOCH: 3/20
Train loss: 0.496988, Train accuracy: 0.7483
[INFO] EPOCH: 4/20
Train loss: 0.486490, Train accuracy: 0.7497
[INFO] EPOCH: 5/20
Train loss: 0.481386, Train accuracy: 0.7513
[INFO] EPOCH: 6/20
Train loss: 0.475738, Train accuracy: 0.7627
[INFO] EPOCH: 7/20
Train loss: 0.470061, Train accuracy: 0.7663
[INFO] EPOCH: 8/20
Train loss: 0.465895, Train accuracy: 0.7703
[INFO] EPOCH: 9/20
Train loss: 0.461488, Train accuracy: 0.7693
[INFO] EPOCH: 10/20
Train loss: 0.454265, Train accuracy: 0.7713
[INFO] EPOCH: 11/20
Train loss: 0.447329, Train accuracy: 0.7800
[INFO] EPOCH: 12/20
Train loss: 0.444302, Train accuracy: 0.7820
[INFO] EPOCH: 13/20
Train loss: 0.446185, Train accuracy: 0.7843
[INFO] EPOCH: 14/20
Train loss: 0.440029, Train accuracy: 0.7797
[INFO] EPOCH: 15/20
Train loss: 0.435587, Train accuracy: 0.7937
[INFO] EPOCH: 16/20
Train loss: 0.

In [9]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP_2_layer(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="envelope.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-3, optim="adam", epochs=30)

loops.test(model_path="envelope.pth", test_loader=test_loader)

[INFO] EPOCH: 1/30
Train loss: 0.566497, Train accuracy: 0.6753
[INFO] EPOCH: 2/30
Train loss: 0.500878, Train accuracy: 0.7343
[INFO] EPOCH: 3/30
Train loss: 0.484213, Train accuracy: 0.7473
[INFO] EPOCH: 4/30
Train loss: 0.466637, Train accuracy: 0.7607
[INFO] EPOCH: 5/30
Train loss: 0.450846, Train accuracy: 0.7747
[INFO] EPOCH: 6/30
Train loss: 0.452504, Train accuracy: 0.7743
[INFO] EPOCH: 7/30
Train loss: 0.440754, Train accuracy: 0.7873
[INFO] EPOCH: 8/30
Train loss: 0.433268, Train accuracy: 0.7890
[INFO] EPOCH: 9/30
Train loss: 0.424627, Train accuracy: 0.7980
[INFO] EPOCH: 10/30
Train loss: 0.428517, Train accuracy: 0.7863
[INFO] EPOCH: 11/30
Train loss: 0.418402, Train accuracy: 0.7980
[INFO] EPOCH: 12/30
Train loss: 0.406408, Train accuracy: 0.8087
[INFO] EPOCH: 13/30
Train loss: 0.398468, Train accuracy: 0.8057
[INFO] EPOCH: 14/30
Train loss: 0.400714, Train accuracy: 0.8040
[INFO] EPOCH: 15/30
Train loss: 0.391444, Train accuracy: 0.8050
[INFO] EPOCH: 16/30
Train loss: 0.

In [10]:
data.X_reduced =  np.hstack((data.X_reduced, data_zcr.X_reduced))

print(data.get_samples().shape)
print(data.get_labels().shape)


#concatenate rms with zcr

(3309, 2252)
(3309,)


In [13]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP_3_layer(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="envelope.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-4, weight_decay=1e-3, optim="adam", epochs=20)

loops.test(model_path="envelope.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.648050, Train accuracy: 0.6737
[INFO] EPOCH: 2/20
Train loss: 0.491434, Train accuracy: 0.7817
[INFO] EPOCH: 3/20
Train loss: 0.444101, Train accuracy: 0.7990
[INFO] EPOCH: 4/20
Train loss: 0.426957, Train accuracy: 0.8017
[INFO] EPOCH: 5/20
Train loss: 0.414838, Train accuracy: 0.8073
[INFO] EPOCH: 6/20
Train loss: 0.394372, Train accuracy: 0.8243
[INFO] EPOCH: 7/20
Train loss: 0.379820, Train accuracy: 0.8343
[INFO] EPOCH: 8/20
Train loss: 0.370578, Train accuracy: 0.8360
[INFO] EPOCH: 9/20
Train loss: 0.354732, Train accuracy: 0.8440
[INFO] EPOCH: 10/20
Train loss: 0.336241, Train accuracy: 0.8597
[INFO] EPOCH: 11/20
Train loss: 0.325302, Train accuracy: 0.8617
[INFO] EPOCH: 12/20
Train loss: 0.302227, Train accuracy: 0.8720
[INFO] EPOCH: 13/20
Train loss: 0.299041, Train accuracy: 0.8747
[INFO] EPOCH: 14/20
Train loss: 0.274919, Train accuracy: 0.8840
[INFO] EPOCH: 15/20
Train loss: 0.261247, Train accuracy: 0.8953
[INFO] EPOCH: 16/20
Train loss: 0.

In [14]:
data_env = Extract_Features(
    df_X=df_X,
    df_Y=df_Y,
    feature = "amplitude_envelope",
    frame_size = 128,
    hop_length = 32
)

print(data_env.get_samples().shape)
print(data_env.get_labels().shape)

(3309, 1122)
(3309,)


In [15]:
data.X_reduced =  np.hstack((data.X_reduced, data_env.X_reduced))

print(data.get_samples().shape)
print(data.get_labels().shape)

(3309, 3374)
(3309,)


In [17]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP_2_layer(nb_hidden=1024, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="envelope.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-4, weight_decay=1e-3, optim="adam", epochs=20)

loops.test(model_path="envelope.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.581179, Train accuracy: 0.7003
[INFO] EPOCH: 2/20
Train loss: 0.457205, Train accuracy: 0.7870
[INFO] EPOCH: 3/20
Train loss: 0.423409, Train accuracy: 0.8027
[INFO] EPOCH: 4/20
Train loss: 0.400227, Train accuracy: 0.8170
[INFO] EPOCH: 5/20
Train loss: 0.394964, Train accuracy: 0.8080
[INFO] EPOCH: 6/20
Train loss: 0.378214, Train accuracy: 0.8280
[INFO] EPOCH: 7/20
Train loss: 0.361274, Train accuracy: 0.8393
[INFO] EPOCH: 8/20
Train loss: 0.348195, Train accuracy: 0.8407
[INFO] EPOCH: 9/20
Train loss: 0.341420, Train accuracy: 0.8500
[INFO] EPOCH: 10/20
Train loss: 0.315618, Train accuracy: 0.8550
[INFO] EPOCH: 11/20
Train loss: 0.294282, Train accuracy: 0.8733
[INFO] EPOCH: 12/20
Train loss: 0.277540, Train accuracy: 0.8787
[INFO] EPOCH: 13/20
Train loss: 0.267641, Train accuracy: 0.8867
[INFO] EPOCH: 14/20
Train loss: 0.246918, Train accuracy: 0.9013
[INFO] EPOCH: 15/20
Train loss: 0.222736, Train accuracy: 0.9057
[INFO] EPOCH: 16/20
Train loss: 0.